# Flow map — 2025 one-way OD, member vs. casual

Goal: render the hub-and-spoke pattern as arcs on a Toronto basemap.

**Important data gotcha discovered during this notebook:** the `End_Station_Name` column in the ridership CSVs is unreliable — it frequently echoes `Start_Station_Name` instead of the real destination. But `End_Station_Id` is correct. So we join by **ID** against the GBFS station feed to recover the true destination name + lat/lon. `Start_Station_Name` is fine, but we use IDs for symmetry.

Output: `data/processed/flow_map.html` (standalone, open in browser).

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / 'src'))

import duckdb
import pandas as pd
import pydeck as pdk

DATA_RAW = Path.cwd().parent / 'data' / 'raw'
DATA_PROC = Path.cwd().parent / 'data' / 'processed'
DATA_PROC.mkdir(parents=True, exist_ok=True)

GLOB = str(DATA_RAW / 'ridership' / '2025' / '*.csv')
con = duckdb.connect()
con.execute(f"CREATE VIEW trips AS SELECT * FROM read_csv_auto('{GLOB}', header=True, ignore_errors=true, union_by_name=true)")

stations = pd.read_parquet(DATA_RAW / 'stations.parquet')
stations['station_id'] = stations['station_id'].astype(str)
len(stations), stations.columns.tolist()

(1030,
 ['station_id',
  'name',
  'lat',
  'lon',
  'capacity',
  'physical_configuration',
  'address',
  'cross_street',
  'post_code',
  'is_charging_station',
  'rental_methods'])

## 1. Canonical station lookup by name

GBFS has one row per `station_id`. Multiple IDs can share a name (dual-docks / re-IDs). Pick the centroid of all IDs that share a name — keeps the map point stable if a station was split.

In [2]:
# Station lookup by ID. GBFS station_id is a string; trips store numeric. Cast once here.
stations['station_id_int'] = pd.to_numeric(stations['station_id'], errors='coerce').astype('Int64')
station_lookup = stations.dropna(subset=['station_id_int']).set_index('station_id_int')[['name', 'lat', 'lon']]
print('Stations indexed by id:', len(station_lookup))
station_lookup.head()

Stations indexed by id: 1030


,name,lat,lon
station_id_int,,,
7000,Fort York Blvd / Capreol Ct,43.639832,-79.395954
7001,Wellesley Station Green P,43.664964,-79.383550
7002,St. George St / Bloor St W,43.667131,-79.399555
7003,Madison Ave / Bloor St W,43.667018,-79.402796
7005,King St W / York St,43.648001,-79.383177


## 2. Clean one-way OD aggregation

Join by station name (not id) and drop same-name pairs.

In [3]:
# Aggregate OD by ID — never trust End_Station_Name.
raw = con.execute('''
    SELECT User_Type AS user_type,
           Start_Station_Id AS o_id,
           End_Station_Id   AS d_id,
           COUNT(*) AS trips
    FROM trips
    WHERE Start_Station_Id IS NOT NULL
      AND End_Station_Id IS NOT NULL
      AND User_Type IN ('Member', 'Casual')
    GROUP BY 1, 2, 3
''').fetchdf()

print('Total pairs (incl. round-trips by id):', len(raw), 'trips:', raw['trips'].sum())

# Attach names + coords for both ends via GBFS lookup.
od = raw.join(station_lookup.add_prefix('o_'), on='o_id', how='inner') \
        .join(station_lookup.add_prefix('d_'), on='d_id', how='inner')

# Drop true round-trips (same station id).
od = od[od['o_id'] != od['d_id']].copy()
od.rename(columns={'o_name': 'origin', 'd_name': 'dest'}, inplace=True)

print('One-way pairs after GBFS join:', len(od), 'trips retained:', od['trips'].sum())
od.sort_values('trips', ascending=False).head(10)[['user_type', 'origin', 'dest', 'trips']]

Total pairs (incl. round-trips by id): 591320 trips: 7805759


One-way pairs after GBFS join: 558964 trips retained: 7126858


,user_type,origin,dest,trips
51233,Member,King St W / Charlotte St,King St W / Bay St (West Side),1342
305318,Member,King St W / Portland St,King St W / Bay St (West Side),1334
299420,Member,Ontario St / Adelaide St E -SMART,Richmond St E / Yonge St,1211
424497,Member,Front St W / Blue Jays Way,Union Station,1136
363143,Member,York St / Queens Quay W,Bathurst St/Queens Quay(Billy Bishop Airport),1075
426608,Member,Frederick St / King St E,King St E / Victoria St,1045
77846,Member,Fort York Blvd / Capreol Ct,Bremner Blvd / Rees St,1009
314284,Member,Cherry Beach,Tommy Thompson Park (Leslie Street Spit),984
53388,Member,Bathurst St/Queens Quay(Billy Bishop Airport),York St / Queens Quay W,969
11,Member,Grand Avenue Park,Windsor St / Newcastle St,945


## 3. Bidirectional pair collapse

For the map we want A↔B shown as one arc (directed by net flow), and we only want the head of the distribution or the map is illegible.

In [4]:
def collapse(group: pd.DataFrame) -> pd.DataFrame:
    g = group.copy()
    g['a_id'] = g[['o_id', 'd_id']].min(axis=1)
    g['b_id'] = g[['o_id', 'd_id']].max(axis=1)
    grouped = g.groupby(['a_id', 'b_id'], as_index=False)['trips'].sum()
    grouped = grouped.join(station_lookup.add_prefix('a_'), on='a_id') \
                     .join(station_lookup.add_prefix('b_'), on='b_id')
    return grouped.dropna(subset=['a_lat', 'b_lat'])

pairs_all = collapse(od)
pairs_casual = collapse(od[od['user_type'] == 'Casual'])
pairs_member = collapse(od[od['user_type'] == 'Member'])

print('Distinct pairs — all:', len(pairs_all), 'casual:', len(pairs_casual), 'member:', len(pairs_member))
pairs_all.sort_values('trips', ascending=False).head(10)[['a_name', 'b_name', 'trips']]

Distinct pairs — all: 205040 casual: 157059 member: 182998


,a_name,b_name,trips
39248,York St / Queens Quay W,Bathurst St/Queens Quay(Billy Bishop Airport),3551
172540,Windsor St / Newcastle St,Grand Avenue Park,3229
132257,Cherry Beach,Tommy Thompson Park (Leslie Street Spit),2777
7629,King St W / Bay St (West Side),King St W / Portland St,2389
39613,York St / Queens Quay W,Coronation Park (Martin Goodman Trail),2372
13,Fort York Blvd / Capreol Ct,Bremner Blvd / Rees St,2258
7363,King St W / Bay St (West Side),King St W / Charlotte St,2231
7953,Bay St / Queens Quay W (Ferry Terminal),Bathurst St/Queens Quay(Billy Bishop Airport),2207
3240,Bay St / College St (East Side),College St / Huron St,2046
128943,King St E / Victoria St,Frederick St / King St E,2012


## 4. Render arc map

Three layers (member / casual / all) so we can eyeball the commute-vs-leisure geography difference directly.

In [5]:
TOP_N = 300

def arcs(pairs: pd.DataFrame, color: list, n: int = TOP_N) -> pdk.Layer:
    top = pairs.sort_values('trips', ascending=False).head(n).copy()
    tmin, tmax = top['trips'].min(), top['trips'].max()
    top['width'] = 0.5 + 5.5 * (top['trips'] - tmin) / max(tmax - tmin, 1)
    top['pair'] = top['a_name'].astype(str) + ' <-> ' + top['b_name'].astype(str)
    return pdk.Layer(
        'ArcLayer',
        data=top,
        get_source_position=['a_lon', 'a_lat'],
        get_target_position=['b_lon', 'b_lat'],
        get_width='width',
        get_source_color=color,
        get_target_color=color,
        pickable=True,
        auto_highlight=True,
    )

member_layer = arcs(pairs_member, [50, 130, 255, 160])
casual_layer = arcs(pairs_casual, [255, 140, 40, 160])

station_layer = pdk.Layer(
    'ScatterplotLayer',
    data=stations[['name', 'lat', 'lon']],
    get_position=['lon', 'lat'],
    get_radius=40,
    get_fill_color=[200, 200, 200, 200],
    pickable=False,
)

view = pdk.ViewState(latitude=43.655, longitude=-79.385, zoom=12, pitch=40)

deck = pdk.Deck(
    layers=[station_layer, member_layer, casual_layer],
    initial_view_state=view,
    map_style='light',
    tooltip={'text': '{pair}\n{trips} trips'},
)
out = DATA_PROC / 'flow_map.html'
deck.to_html(str(out), notebook_display=False)
print('Wrote:', out)

Wrote: /Users/tyler/src/open_data_toronto/bikeshare-od-flows/data/processed/flow_map.html


## 5. Sanity-check: top pairs, cleaned

In [6]:
pd.concat([
    pairs_member.sort_values('trips', ascending=False).head(10).assign(kind='member'),
    pairs_casual.sort_values('trips', ascending=False).head(10).assign(kind='casual'),
])[['kind', 'a_name', 'b_name', 'trips']]

,kind,a_name,b_name,trips
36813,member,York St / Queens Quay W,Bathurst St/Queens Quay(Billy Bishop Airport),2044
7231,member,King St W / Bay St (West Side),King St W / Portland St,2014
121055,member,Cherry Beach,Tommy Thompson Park (Leslie Street Spit),1924
6974,member,King St W / Bay St (West Side),King St W / Charlotte St,1864
118106,member,King St E / Victoria St,Frederick St / King St E,1830
156114,member,Windsor St / Newcastle St,Grand Avenue Park,1680
3043,member,Bay St / College St (East Side),College St / Huron St,1642
38042,member,College St / Major St,Robert St / Bloor St W - SMART,1596
17167,member,Union Station,Front St W / Blue Jays Way,1573
644,member,Wellesley Station Green P,Sherbourne St / Wellesley St E,1556
